# Análisis exploratorio de anuncios de vehículos usados

**Sprint 7 — Herramientas de desarrollo de software**

Este notebook explora el conjunto de datos `vehicles_us.csv` para fundamentar el dashboard web. Se priorizan la calidad de los datos, las distribuciones y la relación descriptiva entre precio y kilometraje.

## 1. Carga y reproducibilidad

Se utiliza una ruta relativa desde el directorio `notebooks` para facilitar la ejecución dentro del repositorio.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

car_data = pd.read_csv(Path('../vehicles_us.csv'))
car_data.head()

In [ ]:
print(f'Filas: {car_data.shape[0]:,}')
print(f'Columnas: {car_data.shape[1]}')
car_data.info()

## 2. Calidad de los datos

Se comprueban duplicados y valores ausentes. No se imputan indiscriminadamente atributos desconocidos: hacerlo podría distorsionar las distribuciones. En cada visualización se excluyen solo las filas donde falta la variable necesaria.

In [ ]:
missing = (car_data.isna().sum().to_frame(name='valores_ausentes')
           .assign(porcentaje=lambda x: (100 * x['valores_ausentes'] / len(car_data)).round(2))
           .sort_values('valores_ausentes', ascending=False))
print(f'Filas duplicadas exactas: {car_data.duplicated().sum()}')
missing

El campo `is_4wd` presenta valores ausentes. No se interpreta automáticamente que todos ellos signifiquen ausencia de tracción 4x4, porque la fuente no documenta ese supuesto.

## 3. Estadísticas descriptivas

Además de la media se utiliza la mediana, una medida más robusta ante precios anunciados excepcionalmente elevados.

In [ ]:
car_data[['price', 'odometer', 'model_year', 'days_listed']].describe()

In [ ]:
print('Precio mediano (USD):', car_data['price'].median())
print('Kilometraje mediano (millas):', car_data['odometer'].median())
print('Tipos de vehículo más frecuentes:')
print(car_data['type'].value_counts().head(5))

## 4. Distribución del kilometraje

El histograma muestra dónde se concentran los vehículos anunciados y deja ver los valores extremos. Se eliminan únicamente los registros que no tienen kilometraje.

In [ ]:
odo_data = car_data.dropna(subset=['odometer'])
fig_hist = px.histogram(odo_data, x='odometer', nbins=50,
    title='Distribución del kilometraje de vehículos anunciados',
    labels={'odometer': 'Kilometraje (millas)'})
fig_hist.update_layout(yaxis_title='Número de anuncios')
fig_hist.show()

## 5. Relación entre kilometraje y precio

El diagrama de dispersión permite examinar la asociación entre variables numéricas, sin demostrar causalidad. Para evitar sobrecargar el navegador, se toma una muestra aleatoria reproducible de hasta 5.000 anuncios completos.

In [ ]:
scatter_data = car_data.dropna(subset=['price', 'odometer']).copy()
if len(scatter_data) > 5_000:
    scatter_data = scatter_data.sample(n=5_000, random_state=42)
fig_scatter = px.scatter(scatter_data, x='odometer', y='price', color='condition',
    opacity=0.5, render_mode='webgl', title='Precio anunciado frente a kilometraje',
    labels={'odometer': 'Kilometraje (millas)', 'price': 'Precio anunciado (USD)',
            'condition': 'Condición'})
fig_scatter.show()

## 6. Comparación por tipo de vehículo

Se analizan conteos y medianas. Estas cifras descriptivas no controlan por marca, año ni estado; por tanto, no prueban que el tipo del vehículo cause las diferencias de precio.

In [ ]:
summary_by_type = (car_data.groupby('type', dropna=False)
    .agg(anuncios=('price', 'size'),
         precio_mediano_usd=('price', 'median'),
         kilometraje_mediano_millas=('odometer', 'median'))
    .sort_values('anuncios', ascending=False))
summary_by_type.head(10)

## 7. Conclusiones

- El dataset incluye **51.525 anuncios** y **13 variables**.
- El precio mediano anunciado es **9.000 USD** y el kilometraje mediano registrado es **113.000 millas**.
- Los tipos más comunes son **SUV (12.405)**, **truck (12.353)** y **sedan (12.154)**.
- Faltan odómetros en **7.892** anuncios y años de modelo en **3.619**.
- Los precios son valores anunciados, no ventas confirmadas. Las relaciones visuales no establecen causalidad y los valores extremos pueden alterar las medias.

**Decisión para la aplicación:** incluir histograma de kilometraje, dispersión entre precio y kilometraje, filtro por tipo de vehículo y métricas descriptivas; son funcionalidades pertinentes a la consigna.